# NLP Basics Assessment

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ohtar10/icesi-nlp/blob/main/Sesion1/6-practice.ipynb)

En este notebook vamos a poner en práctica algunos de los conceptos vistos en los notebooks anteriores, aplicado a un corpus específico: 
[_An Occurrence at Owl Creek Bridge_](https://en.wikipedia.org/wiki/An_Occurrence_at_Owl_Creek_Bridge) por Ambrose Bierce (1890). Esta historia es de dominio público y el corpus fue obtenido de [Project Gutenberg](https://www.gutenberg.org/ebooks/375.txt.utf-8).

## Referencias
* [NLP - Natural Language Processing With Python](https://www.udemy.com/course/nlp-natural-language-processing-with-python)
* [Natural Language Processing in Action](https://www.manning.com/books/natural-language-processing-in-action)

In [31]:
import pkg_resources
import warnings

warnings.filterwarnings('ignore')

installed_packages = [package.key for package in pkg_resources.working_set]
IN_COLAB = 'google-colab' in installed_packages

In [32]:
#!test '{IN_COLAB}' = 'True' && wget  https://github.com/Ohtar10/icesi-nlp/raw/refs/heads/main/requirements.txt && pip install -r requirements.txt

In [33]:
# RUN THIS CELL to perform standard imports:
#%pip install spacy
#%pip install https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.7.1/en_core_web_sm-3.7.1-py3-none-any.whl

import spacy
nlp = spacy.load('en_core_web_sm')

**1. Creamos el documento desde el archivo `sherlock.txt`**<br>


In [34]:
#!test '{IN_COLAB}' = 'True' && wget  https://github.com/Ohtar10/icesi-nlp/raw/refs/heads/main/Sesion1/owlcreek.txt

In [35]:
with open('./sherlock.txt') as file:
    doc = nlp(file.read())

In [36]:
doc[:36]

I A SCANDAL IN BOHEMIA


I

To Sherlock Holmes she is always _the_ woman I have seldom heard him
mention her under any other name In his eyes she eclipses and

El documento fue cargado exitosamente!

**2. Cuantos tokens hay en el archivo?**

In [37]:
len(doc)

9360

**3. Cuantas oraciones hay en el archivo?**
<br>Pista: Necesitarás una lista primero

In [38]:
sentences = list(doc.sents)
len(sentences)

134

**4. Imprime la segunda oración del documento**
<br> Pista: Los índices comienzan en 0 y el título cuenta como la primera oración.

In [39]:
sentences[1]

To Sherlock Holmes she is always _the_ woman I have seldom heard him
mention her under any other name In his eyes she eclipses and
predominates the whole of her sex

**5. Por cada token en la oración anterior, imprime su `text`, `POS` tag, `dep` tag y `lemma`**
<br>

In [40]:
print("{:20}{:20}{:20}{:20}".format("Text", "POS", "dep", "lemma"))
for token in sentences[1]:
    print(f"{token.text:{20}}{token.pos_:{20}}{token.dep_:{20}}{token.lemma_:{20}}")

Text                POS                 dep                 lemma               
To                  ADP                 prep                to                  
Sherlock            PROPN               compound            Sherlock            
Holmes              PROPN               pobj                Holmes              
she                 PRON                nsubj               she                 
is                  AUX                 ROOT                be                  
always              ADV                 advmod              always              
_                   PUNCT               attr                _                   
the                 DET                 det                 the                 
_                   PRON                compound            _                   
woman               NOUN                attr                woman               
I                   PRON                nsubj               I                   
have                AUX     

### Análisis de Frecuencia
Se encuentras las 15 palabras más comunes excluyendo "stop words"

In [ ]:

from collections import Counter

words_clean = [
    t.text.lower()
    for t in doc
    if not t.is_stop and not t.is_punct and t.text.strip()
]

for i, (word, count) in enumerate(Counter(words_clean).most_common(15), 1):
    print(f"{i:2}. {word:<15} {count}")

 1. holmes          47
 2. said            33
 3. man             22
 4. photograph      21
 5. street          18
 6. know            18
 7. king            17
 8. majesty         16
 9. little          14
10. house           14
11. irene           13
12. adler           13
13. door            13
14. minutes         13
15. fire            12


### Análisis de Entidades
Se identifican y listan las 10 entidades más frecuentes

In [ ]:

entities = [ent.text for ent in doc.ents]
for i, (ent, count) in enumerate(Counter(entities).most_common(10), 1):
    print(f"{i:2}. {ent:<10} {count}")

 1. Holmes     29
 2. one        14
 3. Briony Lodge 10
 4. Irene Adler 9
 5. two        7
 6. three      7
 7. Baker Street 6
 8. Majesty    6
 9. Watson     5
10. German     5


**6. Implementa un matcher llamado *Swimming* que encuentre las ocurrencias de la frase *swimming vigorously* Write a matcher called 'Swimming' that finds**
<br>
Pista: Deberías incluir un patrón`'IS_SPACE': True` entre las dos palabras.

In [48]:
from spacy.matcher import Matcher

matcher = Matcher(nlp.vocab)
# Patrón para encontrar "Sherlock Holmes" o "Mr. Holmes"
pattern = [
    [{'LOWER': 'sherlock'}, {'LOWER': 'holmes'}],
    [{'LOWER': 'mr'}, {'IS_PUNCT': True, 'OP': '?'}, {'LOWER': 'holmes'}]
]

# Añadimos el patrón al matcher
matcher.add("HolmesReference", pattern)


In [49]:
found_matches = matcher(doc)

print(f"Se encontraron {len(found_matches)} referencias a Holmes.")




Se encontraron 11 referencias a Holmes.


**7. Imprime el texto al rededor de cada match encontrado**

In [51]:
match_id, start, end = found_matches[0]
print(doc[start-5:end+5])

BOHEMIA


I

To Sherlock Holmes she is always _the


In [52]:
start, end = found_matches[1][1:]
doc[start-7:end+5]

as I had pictured it from
Sherlock Holmes succinct description but the locality

**8. Imprime la oración que contiene cada match encontrado**

In [53]:
for sentence in sentences:
    for _, start, end in found_matches:
        if sentence.start <= start and sentence.end >= end:
            print(sentence.text, '\n')

To Sherlock Holmes she is always _the_ woman I have seldom heard him
mention her under any other name In his eyes she eclipses and
predominates the whole of her sex 

It was a quarter past six when we left Baker Street and it still
wanted ten minutes to the hour when we found ourselves in Serpentine
Avenue It was already dusk and the lamps were just being lighted as
we paced up and down in front of Briony Lodge waiting for the coming
of its occupant The house was just such as I had pictured it from
Sherlock Holmes succinct description but the locality appeared to be
less private than I expected On the contrary for a small street in a
quiet neighbourhood it was remarkably animated There was a group of
shabbily dressed men smoking and laughing in a corner a
scissorsgrinder with his wheel two guardsmen who were flirting with a
nursegirl and several welldressed young men who were lounging up and
down with cigars in their mouths

You see remarked Holmes as we paced to and fro in front of th